In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
IGNORE_INDEX = 0
HIGH_IMPORTANCE_CLASSES = {2, 3}   # building, road
NUM_CLASSES = 8

HR_PATCH = 256
SCALE = 4
LR_PATCH = HR_PATCH // SCALE  # 64

In [6]:
CKPT_DIR = '/kaggle/input/datasets/nvamsikrishna1/all-checkpoints-1'
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)  # for the new segmenter_best.pth

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class RegionImportanceNet(nn.Module):
    def __init__(self, base_ch=32, scale=SCALE):
        super().__init__()
        self.scale = scale
        self.enc1 = ConvBlock(3, base_ch)
        self.enc2 = ConvBlock(base_ch, base_ch*2)
        self.enc3 = ConvBlock(base_ch*2, base_ch*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_ch*4, base_ch*8)
        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, 2, stride=2)
        self.dec3 = ConvBlock(base_ch*8, base_ch*4)
        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec2 = ConvBlock(base_ch*4, base_ch*2)
        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, 2, stride=2)
        self.dec1 = ConvBlock(base_ch*2, base_ch)
        self.out_conv = nn.Conv2d(base_ch, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        importance_lr = torch.sigmoid(self.out_conv(d1))
        return F.interpolate(importance_lr, scale_factor=self.scale, mode='bilinear', align_corners=False)

class SegUNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, base_ch=48):
        super().__init__()
        self.enc1 = ConvBlock(3, base_ch)
        self.enc2 = ConvBlock(base_ch, base_ch*2)
        self.enc3 = ConvBlock(base_ch*2, base_ch*4)
        self.enc4 = ConvBlock(base_ch*4, base_ch*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_ch*8, base_ch*16)
        self.up4 = nn.ConvTranspose2d(base_ch*16, base_ch*8, 2, stride=2)
        self.dec4 = ConvBlock(base_ch*16, base_ch*8)
        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, 2, stride=2)
        self.dec3 = ConvBlock(base_ch*8, base_ch*4)
        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec2 = ConvBlock(base_ch*4, base_ch*2)
        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, 2, stride=2)
        self.dec1 = ConvBlock(base_ch*2, base_ch)
        self.out_conv = nn.Conv2d(base_ch, num_classes, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out_conv(d1)

def compute_miou(pred_logits, target, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    pred = pred_logits.argmax(1)
    ious = []
    for c in range(num_classes):
        if c == ignore_index: continue
        pred_c, target_c = (pred == c), (target == c)
        inter = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        if union == 0: continue
        ious.append(inter / union)
    return sum(ious) / len(ious) if ious else 0.0

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size=16, num_heads=4):
        super().__init__()
        self.window_size = window_size
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        B, C, H, W = x.shape
        ws = self.window_size
        x_windows = x.view(B, C, H // ws, ws, W // ws, ws).permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, ws*ws, C)
        normed = self.norm(x_windows)
        attn_out, _ = self.attn(normed, normed, normed)
        out = (x_windows + attn_out).view(B, H // ws, W // ws, ws, ws, C).permute(0, 5, 1, 3, 2, 4).contiguous()
        return out.view(B, C, H, W)

class SEBlock(nn.Module):
    def __init__(self, ch, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch // reduction), nn.ReLU(inplace=True),
                                 nn.Linear(ch // reduction, ch), nn.Sigmoid())
    def forward(self, x):
        B, C, _, _ = x.shape
        y = self.pool(x).view(B, C)
        return x * self.fc(y).view(B, C, 1, 1)

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        out = self.act(self.conv1(x))
        return x + self.conv2(out)

class SRBackbone(nn.Module):
    def __init__(self, in_ch=3, feat_ch=64, n_resblocks=4, scale=SCALE):
        super().__init__()
        self.stem = nn.Conv2d(in_ch, feat_ch, 3, padding=1)
        self.body = nn.Sequential(*[ResBlock(feat_ch) for _ in range(n_resblocks)])
        self.up1 = nn.Sequential(nn.Conv2d(feat_ch, feat_ch*4, 3, padding=1), nn.PixelShuffle(2), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.Conv2d(feat_ch, feat_ch*4, 3, padding=1), nn.PixelShuffle(2), nn.ReLU(inplace=True))
    def forward(self, x):
        feat = self.stem(x)
        feat = self.body(feat) + feat
        return self.up2(self.up1(feat))

class AdaptiveSRGenerator(nn.Module):
    def __init__(self, feat_ch=64, window_size=16):
        super().__init__()
        self.backbone = SRBackbone(feat_ch=feat_ch)
        self.heavy_attn = WindowAttention(feat_ch, window_size=window_size, num_heads=4)
        self.heavy_se = SEBlock(feat_ch)
        self.heavy_conv = nn.Conv2d(feat_ch, feat_ch, 3, padding=1)
        self.light_conv = nn.Sequential(nn.Conv2d(feat_ch, feat_ch, 3, padding=1), nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(feat_ch, 3, 3, padding=1)
    def forward(self, lr, importance_map):
        feat = self.backbone(lr)
        heavy = self.heavy_conv(self.heavy_se(self.heavy_attn(feat)))
        light = self.light_conv(feat)
        blended = importance_map * heavy + (1 - importance_map) * light
        out = self.out_conv(blended)
        base = F.interpolate(lr, scale_factor=SCALE, mode='bicubic', align_corners=False)
        return torch.clamp(base + out, 0, 1)

class BaselineSRGenerator(nn.Module):
    def __init__(self, feat_ch=64, window_size=16):
        super().__init__()
        self.backbone = SRBackbone(feat_ch=feat_ch)
        self.attn = WindowAttention(feat_ch, window_size=window_size, num_heads=4)
        self.se = SEBlock(feat_ch)
        self.conv = nn.Conv2d(feat_ch, feat_ch, 3, padding=1)
        self.out_conv = nn.Conv2d(feat_ch, 3, 3, padding=1)
    def forward(self, lr):
        feat = self.backbone(lr)
        feat = self.conv(self.se(self.attn(feat)))
        out = self.out_conv(feat)
        base = F.interpolate(lr, scale_factor=SCALE, mode='bicubic', align_corners=False)
        return torch.clamp(base + out, 0, 1)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

rin = RegionImportanceNet().to(device)
rin.load_state_dict(torch.load(os.path.join(CKPT_DIR, 'rin_best.pth'), map_location=device, weights_only=False)['model_state_dict'])
rin.eval()

baseline_gen = BaselineSRGenerator().to(device)
baseline_ck = torch.load(os.path.join(CKPT_DIR, 'baseline_sr_best.pth'), map_location=device, weights_only=False)
baseline_gen.load_state_dict(baseline_ck['model_state_dict'])
baseline_gen.eval()
print(f"Baseline SR loaded, val PSNR at save time: {baseline_ck['val_psnr']:.2f}dB")

adaptive_gen = AdaptiveSRGenerator().to(device)
adaptive_ck = torch.load(os.path.join(CKPT_DIR, 'adaptive_sr_best.pth'), map_location=device, weights_only=False)
adaptive_gen.load_state_dict(adaptive_ck['model_state_dict'])
adaptive_gen.eval()
print(f"Adaptive SR loaded, val PSNR at save time: {adaptive_ck['val_psnr']:.2f}dB")

for p in rin.parameters(): p.requires_grad = False
for p in baseline_gen.parameters(): p.requires_grad = False
for p in adaptive_gen.parameters(): p.requires_grad = False
print("RIN, baseline, adaptive all loaded and frozen.")

Baseline SR loaded, val PSNR at save time: 31.96dB
Adaptive SR loaded, val PSNR at save time: 29.92dB
RIN, baseline, adaptive all loaded and frozen.


In [ ]:
import torch.optim as optim
import time

seg_model = SegUNet().to(device)
seg_opt = optim.Adam(seg_model.parameters(), lr=1e-3)
seg_sched = optim.lr_scheduler.ReduceLROnPlateau(seg_opt, mode='min', factor=0.5, patience=3)
ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

SEG_EPOCHS = 25
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
seg_ckpt_path = '/kaggle/working/checkpoints/segmenter_best.pth'
best_seg_val = float('inf')

for epoch in range(1, SEG_EPOCHS + 1):
    seg_model.train()
    t0 = time.time()
    running = 0.0
    for batch in train_loader:
        hr = batch['hr'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)

        seg_opt.zero_grad()
        logits = seg_model(hr)
        loss = ce_loss(logits, mask)
        loss.backward()
        seg_opt.step()
        running += loss.item() * hr.size(0)

    train_loss = running / len(train_dataset)

    seg_model.eval()
    val_running = 0.0
    val_miou_sum = 0.0
    with torch.no_grad():
        for batch in val_loader:
            hr = batch['hr'].to(device, non_blocking=True)
            mask = batch['mask'].to(device, non_blocking=True)
            logits = seg_model(hr)
            loss = ce_loss(logits, mask)
            val_running += loss.item() * hr.size(0)
            val_miou_sum += compute_miou(logits, mask) * hr.size(0)

    val_loss = val_running / len(val_dataset)
    val_miou = val_miou_sum / len(val_dataset)
    seg_sched.step(val_loss)

    elapsed = time.time() - t0
    print(f"Epoch {epoch:02d}/{SEG_EPOCHS} | train_loss {train_loss:.4f} | "
          f"val_loss {val_loss:.4f} | val_mIoU {val_miou:.4f} | {elapsed:.1f}s")

    if val_loss < best_seg_val:
        best_seg_val = val_loss
        torch.save({'epoch': epoch, 'model_state_dict': seg_model.state_dict(),
                    'val_loss': val_loss, 'val_miou': val_miou}, seg_ckpt_path)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f}, val_mIoU={val_miou:.4f})")

seg_model.eval()
for p in seg_model.parameters(): p.requires_grad = False
print("Segmenter trained and frozen. Best val_loss:", best_seg_val)

In [9]:
import torch.optim as optim
import time

seg_model = SegUNet().to(device)
seg_opt = optim.Adam(seg_model.parameters(), lr=1e-3)
seg_sched = optim.lr_scheduler.ReduceLROnPlateau(seg_opt, mode='min', factor=0.5, patience=3)
ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

SEG_EPOCHS = 25
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
seg_ckpt_path = '/kaggle/working/checkpoints/segmenter_best.pth'
best_seg_val = float('inf')

for epoch in range(1, SEG_EPOCHS + 1):
    seg_model.train()
    t0 = time.time()
    running = 0.0
    for batch in train_loader:
        hr = batch['hr'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)

        seg_opt.zero_grad()
        logits = seg_model(hr)
        loss = ce_loss(logits, mask)
        loss.backward()
        seg_opt.step()
        running += loss.item() * hr.size(0)

    train_loss = running / len(train_dataset)

    seg_model.eval()
    val_running = 0.0
    val_miou_sum = 0.0
    with torch.no_grad():
        for batch in val_loader:
            hr = batch['hr'].to(device, non_blocking=True)
            mask = batch['mask'].to(device, non_blocking=True)
            logits = seg_model(hr)
            loss = ce_loss(logits, mask)
            val_running += loss.item() * hr.size(0)
            val_miou_sum += compute_miou(logits, mask) * hr.size(0)

    val_loss = val_running / len(val_dataset)
    val_miou = val_miou_sum / len(val_dataset)
    seg_sched.step(val_loss)

    elapsed = time.time() - t0
    print(f"Epoch {epoch:02d}/{SEG_EPOCHS} | train_loss {train_loss:.4f} | "
          f"val_loss {val_loss:.4f} | val_mIoU {val_miou:.4f} | {elapsed:.1f}s")

    if val_loss < best_seg_val:
        best_seg_val = val_loss
        torch.save({'epoch': epoch, 'model_state_dict': seg_model.state_dict(),
                    'val_loss': val_loss, 'val_miou': val_miou}, seg_ckpt_path)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f}, val_mIoU={val_miou:.4f})")

seg_model.eval()
for p in seg_model.parameters(): p.requires_grad = False
print("Segmenter trained and frozen. Best val_loss:", best_seg_val)

NameError: name 'train_loader' is not defined